In [10]:
import os
import sys
import requests
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns   
import math
import scipy.stats as stats
from io import StringIO
from datetime import datetime, timedelta
import warnings 
warnings.filterwarnings("ignore")

In [11]:
# 폰트 깨짐 방지
import platform
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic') 
else:
    
    plt.rc('font', family='NanumGothic')

# 마이너스 기호방지
plt.rcParams['axes.unicode_minus'] = False

In [12]:
articles = pd.read_parquet("articles.parquet")
customers = pd.read_parquet("customers.parquet")
submission = pd.read_parquet("sample_submission.parquet")
transactions = pd.read_parquet("transactions.parquet")

In [13]:
transactions

,customer_id,prediction
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0706016001 0706016002 0372860001 0610776002 07...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0706016001 0706016002 0372860001 0610776002 07...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0706016001 0706016002 0372860001 0610776002 07...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0706016001 0706016002 0372860001 0610776002 07...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0706016001 0706016002 0372860001 0610776002 07...
...,...,...
1371975,ffffbbf78b6eaac697a8a5dfbfd2bfa8113ee5b403e474...,0706016001 0706016002 0372860001 0610776002 07...
1371976,ffffcd5046a6143d29a04fb8c424ce494a76e5cdf4fab5...,0706016001 0706016002 0372860001 0610776002 07...
1371977,ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1...,0706016001 0706016002 0372860001 0610776002 07...
1371978,ffffd7744cebcf3aca44ae7049d2a94b87074c3d4ffe38...,0706016001 0706016002 0372860001 0610776002 07...


In [ ]:
submission

In [ ]:
articles

## 데이터 결합

In [ ]:
# 1. 거래 데이터 + 상품 데이터 조인
df = pd.merge(transactions, articles, on='article_id', how='left')

# 2. 거래/상품 데이터 + 고객 데이터 조인
df = pd.merge(df, customers, on='customer_id', how='left')

# 3. 최종 결과에 Submission 테이블 조인
df = pd.merge(df, submission, on='customer_id', how='left')

print(df.head())

## 값 수정

In [ ]:
df['price'] = df['price'] * 590

df.loc[(df['FN'].isna()) & (df['fashion_news_frequency'] == 'Regularly'), 'FN'] = 1
df.loc[(df['FN'].isna()) & (df['fashion_news_frequency'] == 'Monthly'), 'FN'] = 1

## 칼럼 삭제

In [ ]:
cols_to_drop = [
    'detail_desc', 'postal_code', 'product_code',
    'product_type_no', 'graphical_appearance_no', 'colour_group_code', 
    'perceived_colour_value_id', 'perceived_colour_master_id', 
    'department_no', 'index_code', 'index_group_no', 'section_no', 'garment_group_no'
]

df.drop(columns=cols_to_drop, inplace=True)

print(f"남은 칼럼 수: {len(df.columns)}")

## 결측치 처리

In [ ]:
df['FN'] = df['FN'].fillna(0)

df['Active'] = df['Active'].fillna(0)

df['club_member_status'] = df['club_member_status'].fillna('Unknown')

df['fashion_news_frequency'] = df['fashion_news_frequency'].replace('None', 'NONE')
df['fashion_news_frequency'] = df['fashion_news_frequency'].fillna('NONE')

df = df.dropna(subset=['age'])

## 타입 변환

In [ ]:
# 1. Age (나이): 실수에서 정수로 변환
df['age'] = df['age'].astype(int)

# 2. 범주형 데이터
# 'club_member_status'나 'fashion_news_frequency'처럼 반복되는 글자는 'category' 타입으로 바꾸면 메모리 사용량이 80% 이상 줄어든다고 함
df['club_member_status'] = df['club_member_status'].astype('category')
df['fashion_news_frequency'] = df['fashion_news_frequency'].astype('category')

# 3. ID 칼럼: 문자열로 통일 ('0'이 잘리는 문제를 방지하기 위해 문자열로 고정)
df['article_id'] = df['article_id'].astype(str)
df['customer_id'] = df['customer_id'].astype(str)

# 4. 날짜 데이터: 문자열에서 날짜형으로 변환
df['t_dat'] = pd.to_datetime(df['t_dat'])


df['FN'] = df['FN'].astype(int)
df['Active'] = df['Active'].astype(int)

## 칼럼 생성

In [ ]:
# 1. 구간 설정 (0~19세, 20~29세, ..., 59세~150세)
bins = [0, 20, 30, 40, 50, 60, 150]

# 2. 라벨 설정 (각 구간의 이름)
labels = ['10대', '20대', '30대', '40대', '50대', '60대 이상']

# 3. 새로운 칼럼 생성
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=False)

# 확인
print(df[['age', 'age_group']].head())

## 확인

In [ ]:
df.info()

In [ ]:
missing_counts = df.isnull().sum()

print(missing_counts[missing_counts == 0])

## 저장

In [ ]:
# drop=True: 기존의 뒤섞인 인덱스를 삭제
# inplace=True: 변경 사항을 현재 데이터프레임에 바로 적용
df.reset_index(drop=True, inplace=True)

In [ ]:
# index=False: 저장할 때 인덱스 번호를 파일에 포함하지 않음
df.to_csv('clean_data.csv', index=False)